# Categorising Californian housing blocks using Clustering template

We are going to use the k-means clustering model introduced in DT401.5 to identify useful categories in the [housing districts in California](https://scikit-learn.org/stable/datasets/real_world.html#california-housing-dataset).

For clustering we need several independent variables. These are the variables we are going to use to identify the clusters. For clustering all of these need to be numerical and at least some need to be continuous.

## The data

The data is taken from the 1990 U.S. census. Each row represents a census block group: the smallest geographical unit for which the U.S. Census Bureau publishes sample data. Each block group normally contains a population of 600 to 3,000 people.

Data Information:
- **MedHouseVal**:   The median house value, expressed in hundreds of thousands of dollars (\$100,000).
- **MedInc**:        The median income, expressed in tens of thousands of dollars (\$10,000) in the block group
- **HouseAge**:      The median house age in the block group
- **AveRooms**:      The average number of rooms per household in the block group
- **AveBedrms**:     The average number of bedrooms per household in the block group
- **Population**:    The block group population
- **AveOccup**:      The average number of household members in the block group
- **Latitude**:      The block group latitude
- **Longitude**:     The block group longitude


A household is a group of people residing within a home. Since the average number of rooms and bedrooms in this dataset are provided per household, these columns may take surprisingly large values for block groups with few households and many empty houses, such as vacation resorts.

## What we need to consider for a k-means clustering model

For a clustering model to provide useful groups, we need our independent variables to:

- Be independent of each other. If the independent variables are highly correlated then we only need to use one of them to get the same information.
- Have a similar scale. If some have much smaller values or a smaller range of values than the others they will not be taken into account when building the clustering model

## How to use this notebook

In this notebook all the code needed for the analysis has been written. You will be asked to reflect on the outputs of the code. Those tasks will be in one of two types of boxes:

<div class="alert-info">
<b>Reflection boxes:</b> <br>
These have pointers as to what we should be looking for or comments on how to approach the stages of the template with this dataset.
</div>

<div class="alert-success">
<b>Question boxes:</b> <br>
These have questions you should answer before progressing.
</div>

In both cases, if you wish to add your answers to this notebook as you progress, add an extra `markdown` cell to contain your answers. 

If you wish to save your answers at the end of the workshop, remember to download the notebook to your computer.

<a id='Contents'></a>
## Model building steps
To build the model, we will:<b>

- [Import](#import) packages and load in some data 
- [Prepare](#prepare) the data so we can explore it
- [Explore](#explore) the data and make our testable hypothesis
- [Build](#build) the model 
- [Interpret](#interpret) the model results

<div class="alert alert-block alert-warning">
<b>Reminder:</b> <br>
You don't need to understand all the code here for now, just look for:
<ul>
<li> What we are trying to do with each code cell. How does it fit in our objective?
<li> What the outputs of each code cell tell us? Are we reading too much into the results of each code cell?
</ul>
</div>

<a id="import"></a>
## Import packages and read data
[Back to Contents](#Contents)

Let's start by importing the Python packages we will need:
- [**pandas**](https://pandas.pydata.org/): a tabular data manipulation package
- [**seaborn**](https://seaborn.pydata.org/): a data visualisation package
- [**numpy**](https://numpy.org/): a numerical analysis package
- [**scikit-learn**](https://scikit-learn.org/): a model building package

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 

import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.datasets import fetch_california_housing

sns.set_style("whitegrid")

We can now import the data using the Pandas function [`read_csv`](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) and have a look at the first few rows using the method [`head`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.head.html)

In [ ]:
independent_data, dependent_data = fetch_california_housing(return_X_y=True, as_frame=True)
data = independent_data.merge(dependent_data, left_index=True, right_index=True)
data.head()

<a id="prepare"></a>
### Prepare the data
[Back to Contents](#Contents)

For this notebook we are assuming the data has been prepared before being loaded in. However, it is always important to check that you have what you expect.

We can look at what kind of data our table contains using the [`info`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.iloc.html) method. 

We should be asking ourselves:

- Does the data contain the columns I expect?
- Do the columns have the data type I would expect?
- Do any of the columns have missing values? Are these in any columns we intend to use? There cannot be any null values in the rows we plan to use in our model.

In [ ]:
data.info()

If we have any columns with null values that we want to use in our model, we will need to drop those pieces of data. Pandas gives us the [`dropna`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html) method to do this. If you only need to drop null values from one column you can use the `subset` parameter to pass a list of the columns you wish to be checked for null values.

This piece of code has been commented out, so it will not run. To use this, remove the `#` from the start of the line and replace `'column name'` with the column that you wish to be checked for null values. You can examine more than one column at a time by listing all the columns in the square brackets.

<div class="alert-info">
<b>We don't need to drop any rows:</b> <br>
We can see from the output of the <tt>info</tt> method that there are no null values in any of the columns. We can move on to the next step.
</div>

In [ ]:
# data = data.dropna(subset=['column name', ])

If one of the columns is only useful as a unique row identifier, we can use it as an index. We can set it using the [`set_index`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.set_index.html) method. Replace `'index column name'` with the column that you wish to set as the index. If you need this, remove the `#` to uncomment the code.

<div class="alert-info">
<b>We don't need to set any rows as the index:</b> <br> <br>
All of the columns in this dataset contain useful information for our model and none of them are unique identifiers. We can move on to the next step.
</div>

In [ ]:
#data = data.set_index('ID column name')

---
<a id="explore"></a>
### Exploring the data
[Back to Contents](#Contents)

Now that we have checked our data is clean, we can explore it. A good starting point is to look at the descriptive statistics and check that they seem reasonable. We can do so using the [`describe`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html) method.

<div class="alert-info">
<b>We should be asking ourselves:</b> <br>

- Does the data have unexpected outliers (looking at the max and min values) that might suggest a data quality issue?
- Is the spread of the data what you would expect?
</div>

In [ ]:
data.describe()

It is helpful to do these checks visually. Histograms will help us see the distribution of a variable and scatter plots will show us the relationship between variables. Seaborn allows us to plot these using the functions [`histplot`](https://seaborn.pydata.org/generated/seaborn.histplot.html) for histograms and [`scatterplot`](https://seaborn.pydata.org/generated/seaborn.scatterplot.html) for scatter plots.

In [ ]:
sns.histplot(data=data, x='MedHouseVal');

In [ ]:
sns.scatterplot(data=data, x='MedInc', y='MedHouseVal');

It can be helpful to plot all the histograms and scatter plots in one go. Seaborn allows us to do this using a [`pairplot`](https://seaborn.pydata.org/generated/seaborn.pairplot.html).

In [ ]:
sns.pairplot(data);

The scatter plots help us to understand what kind of relationships there are between our variables and how strong they are. We can also quantify this using the Pandas method [`corr`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html)

In [ ]:
data.corr(numeric_only=False)

We can visualise the correlation matrix using a heatmap, as we saw in the last workshop. We can use the Seaborn function [`heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html) to do this.

In [ ]:
corr = data.corr(numeric_only=False)
sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)), 
            cmap='coolwarm', vmax=1, vmin=-1, center=0,
            square=True, linewidths=.5, annot=True, fmt=".2f");

<div class="alert-success">
    <b>Exercise</b> <br>
    <p>Explore the correlations between the different columns</p>
    <ul>
        <li> Which variables are most related? Keep in mind that a strong negative correlation is still a strong relationship.
        <li> To create useful clusters, we need to use variables that are not highly correlated with each other. Which variables are the least correlated?
    </ul>
</div>

### Confirm your hypothesis

Now that you have explored your data and seen the relationships in it, you can pick the independent variables you want to use to build your clusters. Remember, for your independent variables to be useful in predicting the clusters, they need to:

- Be independent of each other. 
- Have a similar scale. As this is quite rare, we will need to normalise the data before building the model

#### Normalising

We will need to keep only the columns we want to use for clustering. We can start by listing all the columns available using the [`columns`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.columns.html) parameter.

In [ ]:
data.columns

Our predictor variables, the independent variables, are likely to be a subset of our data, so we will make a list of the ones we wish to use. Replace and remove the variable names as needed.

In [ ]:
independent_variable_names = ['MedInc', 
                              'HouseAge', 
                              'Latitude', 
                              'MedHouseVal']

With this list we can make a subset of the data

In [ ]:
data_subset = data.loc[:, independent_variable_names]

Now we are ready to normalise the data. We will use the scikit-learn [`MinMaxScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html) function. This will transform the minimum of all the variables to 0 and the maximum to 1. 

In [ ]:
normalised_data_table = MinMaxScaler().fit_transform(data_subset[independent_variable_names])

To turn this array into a table we will transform it into a Pandas [`DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) and pass the information from our unnormalised DataFrame. 

In [ ]:
data_normalised = pd.DataFrame(normalised_data_table)
data_normalised.columns = independent_variable_names
data_normalised.index = data_subset.index

We can check what this had done to our data using a pairplot.

In [ ]:
sns.pairplot(data_normalised);

<div class="alert-info">
<b>We should be asking ourselves:</b> <br>

- Are all the variables now on the same scale?
- Are the relationships between the variables still the same?
</div>

---
<a id="build"></a>
## Build the model
[Back to Contents](#Contents)

Now we have our data in a form we can build our k-means clustering model. To build it, we will use the scikit-learn [`KMeans`](https://scikit-learn.org/dev/modules/clustering.html#k-means) function.

We will try some possible numbers of clusters and see how the Sum Squared Errors (SSE) changes. We know that as we increase the number of clusters, and if the center's of the clusters are evenly spread throughout the data, then the
To do so, we will generate a list of possible numbers of clusters from `2` to `15`.

In [ ]:
potential_cluster_number = range(2, 15)

Now we can loop through this list, fitting the model and recording the SSE for each, which is called `inertia_` here.

In [ ]:
Sum_of_squared_distances = []

for k in potential_cluster_number:
    km = KMeans(n_clusters=k, n_init=10).fit(data_normalised)
    Sum_of_squared_distances.append(km.inertia_)

We can plot this to see how it changes.

In [ ]:
plt.plot(potential_cluster_number, Sum_of_squared_distances, 'x-')
plt.xlabel('Number of clusters')
plt.ylabel('Sum of Squared Errors');

<div class="alert-success">
    <b>Exercise</b> <br>
    <p>Look at the plot of clusters vs Sum of Squared Errors (SSE). We are looking for one or more <tt>elbow</tt> in the change of SSE:</p>
    <ul>
        <li> Where does it begin to change more slowly? 
        <li> How many clusters do you think we should use?
    </ul>
    <p> Once you have decided a number of clusters to use, enter it as the <tt>number_of_clusters</tt> in the code cell below. A default value has been set for you, but it can be improved upon.</p>
</div>

In [ ]:
number_of_clusters = 2
cluster_model = KMeans(n_clusters=number_of_clusters, random_state=13, n_init='auto').fit(data_normalised)

We can store the cluster identified for each datapoint alongside the unnormalised data. The model parameter [`labels_`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html) will give us the cluster number for each row.

In [ ]:
data_subset['clusters'] = cluster_model.labels_

---
<a id="interpret"></a>
## Interpret the model results
[Back to Contents](#Contents)

<div class="alert-info">
<b>Now we have identified some potential clusters, we should be asking ourselves:</b> <br>

- Do any of the clusters represent a meaningful group? It may only be one or two that do.
- Are any of the clusters too small to be useful? This might help us identify where more data is needed or any outliers.
- Are any of the clusters too large to be useful? This might help us identify where we need to split the data into more clusters.
</div> 

We can count the number of data points in each cluster by grouping the data in our DataFrame using the [`groupby`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) and [`count`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.count.html) methods.

In [ ]:
data_subset.groupby(['clusters']).count()

We can colour a pairplot with these cluster labels

In [ ]:
sns.pairplot(data_subset, hue='clusters');

We can also examine the location of the centres of all the clusters by reading them from the model using [`cluster_centers_`](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html#sklearn.cluster.KMeans.cluster_centers_) and putting them into a Pandas DataFrame

In [ ]:
pd.DataFrame(cluster_model.cluster_centers_, columns=independent_variable_names)

Lastly, we can use Seaborn to plot [`violinplot`](https://seaborn.pydata.org/generated/seaborn.violinplot.html) to compare the clusters

In [ ]:
for col in independent_variable_names:
    fig, ax = plt.subplots()
    sns.violinplot(data=data_subset, 
                   y=col, 
                   x='clusters',
                   inner='quartile',
                   ax=ax);

<div class="alert-success">
    <b>Exercise</b> <br>
    <p>Look at the plot above showing the differences between the clusters:</p>
    <ul>
        <li> For each variable, how do the clusters differ?
        <li> How can these differences be described?
        <li> Can you use these differences to describe each cluster?
    </ul>
</div>